# URLBERT Adversarial Examples
Attack the URLBERT baseline (MODEL 1) with gradient-guided character edits, check validity with `validate_adv.py`, then evaluate MODEL 1 on `test_original + test_adversarial`. Rules: `attack-rules.md`.

In [1]:
%pip install -q -r ../requirements.txt


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: /opt/pytorch/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os
import re
import json
import string
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertConfig, BertModel, BertTokenizerFast
from tqdm.auto import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # silence fork warning
sys.path.append("../src")
from load_data import load_splits, load_config
from evaluate import (evaluate, adversarial_metrics, deltas, adversarial_rows, adversarial_recall, attack_summary,
                      save_preds)
from validate_adv import SCHEME, parts, edit_distance, invalid_reason

/opt/pytorch/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# Setup
- `SMOKE = True`: about 75 URLs with the smoke checkpoint (local check)
- `SPLIT`: `test` → `test_adversarial` (`urlbert_generate_examples_test.ipynb`), `train` → `train_adversarial` (`urlbert_generate_examples_train.ipynb`)
- `FRAC`: share of the phishing URLs MODEL 1 detects that get attacked (same seed → a larger `FRAC` contains this sample)

In [3]:
SMOKE = False
SPLIT = "train"
FRAC = 0.5 if SMOKE else 0.25  # share of MODEL 1-detected phishing urls to attack
K = load_config()[0]["budget"]  # char edit budget (3)
TOP_POS = 3  # tokens to try edits in per round, picked by gradient
BATCH = 64  # urls attacked together
SEED = 42

TAG = "urlbert_smoke" if SMOKE else "urlbert"
URLBERT_DIR = "../models/urlbert"
RESULTS = "../results"
CKPT = "../checkpoints/urlbert_smoke_model1.pt" if SMOKE else "../models/urlbert_baseline.pt"  # MODEL 1
OUT = f"{RESULTS if SMOKE else '../data'}/{TAG}_adv_{SPLIT}.csv"  # smoke outputs stay out of data/
PARTIAL = OUT.replace(".csv", "_partial.csv")

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
use_amp = device == "cuda"
print("device:", device, "| split:", SPLIT, "| k =", K)

device: cuda | split: train | k = 3


# Load Data and Model

In [4]:
splits = load_splits()
tokenizer = BertTokenizerFast(vocab_file=f"{URLBERT_DIR}/bert_tokenizer/vocab.txt", do_lower_case=True)

#same MAX_LEN as training (p99 of train token length, capped at 128)
lengths = [len(x) for x in tokenizer(splits["train"].URL.tolist())["input_ids"]]
MAX_LEN = min(int(np.percentile(lengths, 99)), 128)
print("MAX_LEN:", MAX_LEN)

MAX_LEN: 85


- same classifier as `urlbert_training`, weights frozen (only the input gets a gradient)

In [5]:
class URLBERTClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.fc(self.dropout(h))


config = BertConfig.from_json_file(f"{URLBERT_DIR}/bert_config/config.json")
config.vocab_size = 5000
model = URLBERTClassifier(BertModel(config, add_pooling_layer=False))
model.load_state_dict(torch.load(CKPT, map_location="cpu", weights_only=True))  # strict
model.to(device).eval()
for p in model.parameters():
    p.requires_grad_(False)
emb = model.encoder.embeddings.word_embeddings.weight  # (vocab, 768)


@torch.no_grad()
def p_phish(urls, bs=1024):
    # P(phishing) for a list of URLs; scored shortest first so each batch pads to a similar length
    order = np.argsort([len(u) for u in urls], kind="stable")
    out = np.zeros(len(urls))
    for i in range(0, len(urls), bs):
        idx = order[i:i + bs]
        enc = tokenizer([urls[j] for j in idx], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(enc.input_ids, enc.attention_mask).float()
        out[idx] = torch.softmax(logits, -1)[:, 0].cpu().numpy()
    return out

## Who Gets Attacked
- phishing URLs (label 0) that MODEL 1 detects (`p_phish >= 0.5`), missed ones are skipped

In [6]:
phish = splits[SPLIT][splits[SPLIT].label == 0]
if SMOKE:
    phish = phish.sample(300, random_state=SEED)
p_all = p_phish(phish.URL.tolist())
df = phish[p_all >= 0.5].assign(p_clean=p_all[p_all >= 0.5])
df = df.sample(frac=FRAC, random_state=SEED)
print(f"phishing: {len(phish)} | missed by MODEL 1: {(p_all < 0.5).sum()} | to attack: {len(df)}")

phishing: 77552 | missed by MODEL 1: 15650 | to attack: 15476


# Attack
Gradient-guided greedy character edits (HotFlip-style). Each round (up to `K`):
1. **gradient**: loss w.r.t. the input embeddings, HotFlip score per token = `max_v (e_v - e_i) · grad_i` (how much swapping token i could raise the loss)
2. pick the `TOP_POS` highest-scoring tokens outside the registered domain
3. try every 1-char substitute / insert / delete inside them, plus appending a char (or `/` + char); keep only valid ones within budget (`invalid_reason`)
4. score all candidates exactly, keep the one with the lowest `p_phish`

Stops early once `p_phish < 0.5` or no edit lowers it. URLs are attacked `BATCH` at a time in lockstep (same result as one at a time, just faster on GPU).

In [7]:
HOST_CHARS = string.ascii_lowercase + string.digits + "-."
PATH_CHARS = HOST_CHARS + "_~/=&?#"


def token_scores(urls):
    # hotflip score per token + its char span, for a batch of urls (each url's gradient only depends on its own loss)
    enc = tokenizer(urls, truncation=True, max_length=MAX_LEN, padding=True, return_offsets_mapping=True, return_tensors="pt")
    ids, mask = enc.input_ids.to(device), enc.attention_mask.to(device)
    x = emb[ids].detach().requires_grad_()
    h = model.encoder(inputs_embeds=x, attention_mask=mask).last_hidden_state[:, 0]
    loss = F.cross_entropy(model.fc(h), torch.zeros(len(urls), dtype=torch.long, device=device), reduction="sum")  # true label = phishing
    loss.backward()  # gradient ascent: we want edits that increase this loss
    g = x.grad
    score = (g @ emb.T).max(-1).values - (g * x).sum(-1)  # (batch, tokens)
    return score.tolist(), enc.offset_mapping.tolist()  # padding/special tokens have span (0, 0)


def domain_span(url):
    # char span [start, end) of the registered domain (frozen), end is also where the hostname ends
    host, reg, *_ = parts(url)
    m = SCHEME.match(url)
    i = m.end() if m else 0
    i += re.match(r"[^/?#]*", url[i:]).group().rfind("@") + 1  # skip userinfo
    return i + len(host) - len(reg), i + len(host)


def candidates(url, spans):
    # every 1-char edit inside the given char spans, plus appending a char or '/' + char (a trailing '/' alone isn't canonical)
    a, b = domain_span(url)
    out = {url + c for c in PATH_CHARS} | {url + "/" + c for c in PATH_CHARS}
    for s, e in spans:
        for i in range(s, e + 1):
            chars = HOST_CHARS if i < b else PATH_CHARS
            out |= {url[:i] + c + url[i:] for c in chars}  # insert
            if i < e and not a <= i < b:
                out |= {url[:i] + c + url[i + 1:] for c in chars}  # substitute
                out.add(url[:i] + url[i + 1:])  # delete
    out.discard(url)
    return list(out)


def attack(urls, ps):
    # greedy attack on a batch: up to K rounds of the best edit per url, returns the adversarial urls
    adv, p, done = list(urls), list(ps), [False] * len(urls)
    for _ in range(K):
        live = [i for i in range(len(urls)) if not done[i] and p[i] >= 0.5]
        if not live:
            break
        scores, offs = token_scores([adv[i] for i in live])
        cands = []
        for i, score, off in zip(live, scores, offs):
            a, b = domain_span(adv[i])
            ok = [t for t, (s, e) in enumerate(off) if e > s and not (a <= s and e <= b)]  # skip special + domain tokens
            top = sorted(ok, key=lambda t: -score[t])[:TOP_POS]
            cands.append([c for c in candidates(adv[i], [off[t] for t in top]) if not invalid_reason(urls[i], c, K)])
        p_cands = p_phish([c for cs in cands for c in cs])  # one scoring pass for the whole batch
        j = 0
        for i, cs in zip(live, cands):
            pc, j = p_cands[j:j + len(cs)], j + len(cs)
            if len(cs) and pc.min() < p[i]:
                adv[i], p[i] = cs[pc.argmin()], pc.min()
            else:
                done[i] = True  # no edit lowers p_phish
    return adv

## Generate
- progress is appended to `PARTIAL` after every batch; rerunning the notebook resumes from it (deleted once the final csv is saved)
- columns (see `attack-rules.md`): `orig_URL, URL, label, p_clean, p_adv, n_edits, flipped, duplicate`

In [ ]:
if os.path.exists(PARTIAL):
    done = pd.read_csv(PARTIAL, keep_default_na=False)
    print("resuming:", len(done), "URLs already attacked")
else:
    done = pd.DataFrame(columns=["orig_URL", "URL"])
todo = df[~df.URL.isin(done.orig_URL)]

t0 = time.time()
for i in tqdm(range(0, len(todo), BATCH)):
    chunk = todo.iloc[i:i + BATCH]
    adv = attack(chunk.URL.tolist(), chunk.p_clean.tolist())
    pd.DataFrame({"orig_URL": chunk.URL.values, "URL": adv}).to_csv(PARTIAL, mode="a", header=not os.path.exists(PARTIAL), index=False)
sec_per_url = (time.time() - t0) / max(1, len(todo))
print(f"{sec_per_url:.3f}s per URL")

adv_map = dict(pd.read_csv(PARTIAL, keep_default_na=False).values)
adv_df = pd.DataFrame({"orig_URL": df.URL.values, "URL": df.URL.map(adv_map).values, "label": 0, "p_clean": df.p_clean.values})
adv_df["p_adv"] = p_phish(adv_df.URL.tolist())
adv_df["n_edits"] = [edit_distance(o, a) for o, a in zip(adv_df.orig_URL, adv_df.URL)]
adv_df["flipped"] = adv_df.p_adv < 0.5

resuming: 10496 URLs already attacked


  0%|          | 0/78 [00:00<?, ?it/s]

# Validity Check
- independent re-check with `invalid_reason` (edit budget, registered domain, hostname, characters, port/userinfo, canonical), expect 0 invalid
- `duplicate`: edited URL already in the split or made by an earlier attack; kept for attack metrics, not appended

In [ ]:
reasons = pd.Series([invalid_reason(o, a, K) for o, a in zip(adv_df.orig_URL, adv_df.URL)])
print(reasons.replace("", "valid").value_counts())
assert (reasons == "").all(), "attack produced invalid URLs"

adv_df["duplicate"] = (adv_df.n_edits > 0) & (adv_df.URL.isin(splits[SPLIT].URL) | adv_df.URL.duplicated())

display(adv_df[adv_df.n_edits > 0].sample(min(10, (adv_df.n_edits > 0).sum()), random_state=SEED))

# Summary
- `attack_summary` (`evaluate.py`): flipped / edited-not-flipped / not-edited / duplicates, success rate within 1..k edits, which URL parts were edited

In [ ]:
summary = {
    "split": SPLIT,
    "phishing": len(phish),
    "missed_by_model1": int((p_all < 0.5).sum()),
    **attack_summary(adv_df, K),
    "n_edits": {int(j): int(n) for j, n in adv_df.n_edits.value_counts().sort_index().items()},
    "sec_per_url": round(sec_per_url, 4),
}
assert summary["flipped"] + summary["edited_not_flipped"] + summary["not_edited"] == summary["attacked"]

adv_df.to_csv(OUT, index=False)
os.remove(PARTIAL)  # final csv saved, next run starts fresh
with open(f"{RESULTS}/{TAG}_adv_{SPLIT}_summary.json", "w") as f:
    json.dump(summary, f, indent=1)
summary

# Evaluate MODEL 1 (test only)
- `test_original + test_adversarial` = test set with every edited, non-duplicate URL appended (label 0)
- `adversarial_recall`: recall on the appended adversarial rows alone (comparable across models, unlike delta recall which depends on how many rows were appended)
- white-box: the examples were made against this model, so attack success is expected to be high

In [ ]:
if SPLIT == "test":
    test = splits["test"].sample(500, random_state=SEED) if SMOKE else splits["test"]
    p_test = p_phish(test.URL.tolist())
    if not SMOKE:  # same model + preprocessing as the training notebook
        ref = pd.read_csv(f"{RESULTS}/urlbert_test_original.csv")
        assert (ref.URL.values == test.URL.values).all()
        print("max |p_phish - saved test_original|:", np.abs(ref.p_phish.values - p_test).max())

    edited = adversarial_rows(adv_df)
    both = pd.concat([test[["URL", "label"]], edited[["URL", "label"]]], ignore_index=True)
    p_both = np.concatenate([p_test, edited.p_adv.values])

    m_orig, m_adv = evaluate(test.label, p_test), evaluate(both.label, p_both)
    adv_metrics = adversarial_metrics(adv_df.label, adv_df.p_clean, adv_df.p_adv)  # success rate = flipped / attacked
    adv_metrics["adversarial_recall"] = adversarial_recall(edited.p_adv)
    adv_metrics.update(deltas(m_orig, m_adv))

    save_preds(f"{RESULTS}/{TAG}_test_adversarial.csv", both.URL, both.label, p_both)
    with open(f"{RESULTS}/{TAG}_test_adversarial.json", "w") as f:
        json.dump({**m_adv, **adv_metrics}, f, indent=1)

    display(pd.DataFrame({"test_original": m_orig, "test_original + test_adversarial": m_adv})
            .drop("confusion_matrix").T.astype(float).round(4))
    print({k: round(v, 4) for k, v in adv_metrics.items()})